# 📓 Semana 13 · Dia 2 — Unity AI Gateway: roteamento, fallback e cache semântico

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition (conceito/FMA) + 🔑 avançado (trial) |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate (Governance ~15%) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Gateway configurado no projeto |

---


## 📖 Teoria — Unity AI Gateway (DAIS 2026)

O **Unity AI Gateway** é a camada de controle entre sua aplicação e os LLMs:

- **Roteamento**: enviar cada pergunta ao modelo mais barato/adequado
- **Fallback**: se o modelo principal falhar, chama o reserva
- **Cache semântico**: perguntas similares reutilizam resposta (economia até ~60% de custo)
- **Auditoria**: todas as chamadas logadas (governança)


### 💻 Na prática — Padrão de gateway no código

Implemente o padrão roteamento+fallback no cliente de LLM.


In [ ]:
# Roteamento por tipo de tarefa (padrão gateway)
from mlflow.deployments import get_deploy_client
client = get_deploy_client("databricks")

def chamar_llm(pergunta, tarefa="geral"):
    # Roteamento: tarefas simples -> modelo pequeno/barato; complexas -> grande
    endpoint = ("databricks-llama-3-1-8b" if tarefa == "simples"
                else "databricks-llama-3-1-70b")
    try:
        resp = client.predict(endpoint=endpoint,
                             inputs={"messages": [{"role": "user", "content": pergunta}],
                                     "temperature": 0})
        return resp["choices"][0]["message"]["content"], endpoint
    except Exception as e:
        # Fallback: tenta o modelo reserva
        resp = client.predict(endpoint="databricks-llama-3-1-8b",
                             inputs={"messages": [{"role": "user", "content": pergunta}]})
        return resp["choices"][0]["message"]["content"], "fallback"
print("Padrão roteamento + fallback implementado.")

In [ ]:
# Cache semântico (didático — cache por similaridade de embedding)
from databricks.vector_search.client import VectorSearchClient
cache = {}
def com_cache(pergunta):
    if pergunta in cache:
        print("Cache HIT (resposta reutilizada)")
        return cache[pergunta]
    resp, ep = chamar_llm(pergunta)
    cache[pergunta] = resp
    print(f"Cache MISS (gerou com {ep})")
    return resp
print(com_cache("Qual a receita total?"))
print(com_cache("Qual a receita total?"))  # 2a vez: cache

### 💻 Na prática — Gateway gerenciado (trial)

No trial: **AI → Unity AI Gateway → Create provider route** — configure roteamento/fallback entre modelos e ative cache semântico.


In [ ]:
# Auditoria no gateway
print("""
Todas as chamadas passam a ter: modelo usado, custo, latência, usuário
-> system tables / logs do gateway para auditoria (governança)
""")
print("Na Free, o padrão didático acima mostra o conceito; o gateway gerenciado é 🔑.")

> 🎯 **Dica de prova**: GenAI Assoc (Governance ~15%): gateway cobre roteamento, fallback, cache semântico e auditoria. Pergunta: 'como reduzir custo de LLM?' → cache semântico + roteamento para modelo barato.


## 🎯 Exercícios de fixação

**1.** Explique cache semântico em 2 frases.

**2.** Quando o fallback é acionado?

**3.** Por que auditoria de chamadas importa em empresas?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Cache semântico

Guarda respostas por similaridade de pergunta — perguntas iguais/similares reutilizam a resposta em vez de chamar o LLM (economia de tokens).

**2.** Fallback

Quando o modelo principal falha (erro, quota, latência) — o gateway redireciona para um reserva e mantém o serviço no ar.

**3.** Auditoria

Compliance e custo: saber quem chamou o quê, quanto custou e com qual modelo — obrigatório em empresas reguladas.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*